# SICS: Super-slide

In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial import cKDTree
from IPython.display import display

BASE = Path.cwd()
INPUT = BASE / 'data/pooled_spot_states.h5ad'
OUT = BASE / 'results'
OUT.mkdir(exist_ok=True)
SEED = 186
N_PERMUTATIONS = 50
print({p: version(p) for p in ['anndata', 'numpy', 'pandas', 'scipy']})

In [ ]:
adata = ad.read_h5ad(INPUT)
states = adata.var_names.tolist()
labels = np.asarray(adata.X, dtype=np.int64)
assert np.isin(labels, [0, 1]).all()
assert adata.shape == (33872, 6)
assert len(adata.layers) == 0 and adata.raw is None
assert adata.obs_names.is_unique
assert adata.obs.shape[1] == 0
assert adata.obs_names.tolist() == [f'SPOT_{i:06d}' for i in range(1, adata.n_obs + 1)]
display(pd.DataFrame(labels, index=adata.obs_names, columns=states).head(10))
display(pd.Series(labels.sum(axis=0), index=states, name='positive_spots').to_frame())
print('Pooled spots:', adata.n_obs)

In [ ]:
def build_hex_grid(n_spots, spacing=100):
    if n_spots < 2:
        raise ValueError('At least two sites are required')
    n_rows = int(np.sqrt(n_spots))
    n_cols = n_spots // n_rows + 1
    row, col = np.divmod(np.arange(n_spots), n_cols)
    coordinates = np.column_stack([
        11672 + col * spacing + (row % 2) * (spacing // 2),
        3207 + row * int(spacing * np.sqrt(3) / 2),
    ])
    edges = cKDTree(coordinates).query_pairs(spacing * 1.01, output_type='ndarray')
    edges = np.sort(edges, axis=1)
    edges = edges[np.lexsort((edges[:, 1], edges[:, 0]))]
    graph = sparse.coo_matrix((np.ones(2 * len(edges), dtype=np.int64),
                              (np.r_[edges[:, 0], edges[:, 1]],
                               np.r_[edges[:, 1], edges[:, 0]])),
                             shape=(n_spots, n_spots)).tocsr()
    return coordinates, edges, graph

coordinates, edges, graph = build_hex_grid(len(labels))
assert len(edges) == 100880
assert (graph != graph.T).nnz == 0 and not graph.diagonal().any()
print('Grid sites:', len(coordinates))
print('Undirected spatial edges:', len(edges))

In [ ]:
def count_state_pair_events(labels, edges):
    labels = np.asarray(labels)
    if labels.ndim != 2 or not np.isin(labels, [0, 1]).all():
        raise ValueError('Expected a binary spot-by-state matrix')
    edges = np.asarray(edges)
    if edges.ndim != 2 or edges.shape[1] != 2:
        raise ValueError('Expected two indices per edge')
    if not np.isfinite(edges).all() or not np.equal(edges, np.floor(edges)).all():
        raise ValueError('Edge indices must be finite integers')
    edges = edges.astype(np.int64)
    if edges.size and (edges.min() < 0 or edges.max() >= len(labels)):
        raise ValueError('Edge index outside label table')
    if np.any(edges[:, 0] >= edges[:, 1]) or len(np.unique(edges, axis=0)) != len(edges):
        raise ValueError('Supply each edge once, with source < target')
    present = [np.flatnonzero(row) for row in labels]
    matrix = np.zeros((labels.shape[1], labels.shape[1]), dtype=np.int64)
    for source, target in edges:
        for idx1 in present[source]:
            for idx2 in present[target]:
                matrix[idx1, idx2] += 1
                if idx1 == idx2:
                    continue
                matrix[idx2, idx1] += 1
    return matrix

In [ ]:
np.random.seed(SEED)
placement = np.arange(len(labels))
np.random.shuffle(placement)
draws = np.empty((N_PERMUTATIONS, len(states), len(states)), dtype=np.int64)
for iteration in range(N_PERMUTATIONS):
    np.random.shuffle(placement)
    shuffled_labels = labels[np.argsort(placement)]
    draws[iteration] = count_state_pair_events(shuffled_labels, edges)
    if (iteration + 1) % 10 == 0:
        print(f'Completed {iteration + 1}/{N_PERMUTATIONS} permutations')

matrix = draws.mean(axis=0)
reference = pd.DataFrame(matrix, index=states, columns=states)
connect_sum = float(np.tril(matrix, k=0).sum())
assert np.allclose(matrix, matrix.T)
display(reference)
print('Reference raw-event total:', connect_sum)

In [ ]:
reference.to_csv(OUT / 'UKF_reference_raw.csv')
pd.DataFrame([{'connect_sum_raw': connect_sum}]).to_csv(
    OUT / 'super_slide_connect_sum.csv', index=False)
np.savez_compressed(OUT / 'super_slide_draws.npz', draws=draws,
                    states=np.asarray(states), seed=np.uint64(SEED))
pd.DataFrame(coordinates, columns=['x', 'y']).to_csv(OUT / 'grid_coordinates.csv', index=False)
pd.DataFrame(edges, columns=['source', 'target']).to_csv(OUT / 'grid_edges.csv', index=False)
metadata = {
    'seed': SEED, 'rng': 'MT19937', 'random_api': 'np.random.seed / np.random.shuffle',
    'shuffle_order': 'one initial shuffle, then 50 cumulative shuffles',
    'permutations': N_PERMUTATIONS, 'n_spots': len(labels), 'physical_edges': len(edges),
    'states': states, 'whole_label_rows_permuted': True,
    'counting': 'same-state once, off-diagonal symmetric',
    'total': 'lower triangle including diagonal, without pseudocounts',
    'connect_sum_raw': connect_sum,
    'input_sha256': hashlib.sha256(INPUT.read_bytes()).hexdigest(),
    'reference_sha256': hashlib.sha256((OUT / 'UKF_reference_raw.csv').read_bytes()).hexdigest(),
    'versions': {p: version(p) for p in ['anndata', 'numpy', 'pandas', 'scipy']},
}
(OUT / 'run_manifest.json').write_text(json.dumps(metadata, indent=2) + '\n')
print('Saved results:', OUT.relative_to(BASE))